In [1]:
import sys; sys.path.insert(0, '/home/lsys/private_blacklight/scripts')
import pandas as pd
import janitor  # noqa
import statsmodels.formula.api as smf
from constants import filepaths

TRACKERS = ['ddg_join_ads', 'third_party_cookies', 'canvas_fingerprinting',
            'session_recording', 'key_logging', 'fb_pixel', 'google_analytics']
LABELS = {'ddg_join_ads': 'Ad', 'third_party_cookies': 'Cookies',
          'canvas_fingerprinting': 'Canvas FP', 'session_recording': 'Session',
          'key_logging': 'Keylog', 'fb_pixel': 'FB Pixel', 'google_analytics': 'GA'}

Checking that all paths exist:
{'web_mobile': True, 'web_desktop': True, 'web': True, 'yg_profile': True, 'blacklight': True, 'who': True}


In [2]:
df_visits = (
    pd.concat([
        pd.read_csv(p, usecols=['caseid', 'private_domain', 'device_type'], low_memory=False)
        for p in [
            '/home/lsys/private_blacklight/data/yg/realityMine_web_mobile_2022-06-01_2022-06-30.csv',
            '/home/lsys/private_blacklight/data/yg/realityMine_web_desktop_2022-06-01_2022-06-30.csv',
            '/home/lsys/private_blacklight/data/yg/realityMine_web_2022-06-01_2022-06-30.csv',
        ]
    ], ignore_index=True)
    .dropna(subset=['private_domain'])
)
df_bl = (
    pd.read_csv(filepaths['blacklight'])
    .assign(private_domain=lambda d: d['filename'].str.replace('_', '.', regex=False))
    .remove_columns('filename')
)
df_demo = pd.read_csv(
    '/home/lsys/private_blacklight/data/combined_yg_bl_who.csv',
    usecols=['caseid', 'gender_lab', 'race_lab', 'educ_lab', 'agegroup_lab'],
)

/home/lsys/private_blacklight/bl_venv/lib/python3.10/site-packages/pandas_flavor/register.py:164: FutureWarning: This function will be deprecated in a 1.x release. Please use `pd.DataFrame.drop` instead.
  return method(self._obj, *args, **kwargs)


In [3]:
def aggregate(visits):
    return (
        visits.merge(df_bl, on='private_domain', how='left')
              .dropna(subset=['ddg_join_ads'])
              .groupby('caseid')
              .agg(**{f'bl_{c}': (c, 'sum') for c in TRACKERS},
                   tt_visits=('private_domain', 'size'))
              .assign(**{f'bl_{c}_rate': lambda d, c=c: d[f'bl_{c}'] / d['tt_visits']
                         for c in TRACKERS})
              .reset_index()
              .merge(df_demo, on='caseid')
              .assign(woman=lambda d: (d['gender_lab'] == 'Female').astype(int))
    )

data_full = aggregate(df_visits)
data_desk = aggregate(df_visits.query("device_type == 'Laptop/Desktop'"))
print(f'full: {len(data_full)} users')
print(f'desk: {len(data_desk)} users')

full: 1132 users
desk: 690 users


In [4]:
FORMULA = ("woman + C(race_lab, Treatment('White'))"
           " + C(educ_lab, Treatment('HS or Below'))"
           " + C(agegroup_lab, Treatment('<25'))")


def stars(p):
    return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''


def fit_one(data, y):
    return smf.ols(f'{y} ~ {FORMULA}', data=data).fit(cov_type='HC1')


def coef_table(data, kind):
    suffix = '_rate' if kind == 'rate' else ''
    out = pd.concat([
        (lambda m: m.params.round(3).astype(str) + m.pvalues.apply(stars))(fit_one(data, f'bl_{c}{suffix}')).rename(LABELS[c])
        for c in TRACKERS
    ], axis=1)
    out.index = (out.index
                 .str.replace("C(agegroup_lab, Treatment('<25'))[T.", 'Age:', regex=False)
                 .str.replace("C(race_lab, Treatment('White'))[T.", 'Race:', regex=False)
                 .str.replace("C(educ_lab, Treatment('HS or Below'))[T.", 'Educ:', regex=False)
                 .str.rstrip(']'))
    return out


In [5]:
print('=== Exposure rate, FULL (n={}) ==='.format(len(data_full)))
print(coef_table(data_full, 'rate'))
print()
print('=== Exposure rate, DESKTOP-ONLY (n={}) ==='.format(len(data_desk)))
print(coef_table(data_desk, 'rate'))

=== Exposure rate, FULL (n=1132) ===


                          Ad    Cookies Canvas FP   Session     Keylog  \
Intercept            4.43***    5.69***  0.048***  0.025***   0.028***   
Race:Asian         -1.919***  -2.396***  -0.021**  -0.02***  -0.025***   
Race:Black            -0.027     -0.521     0.005      0.01      0.004   
Race:Hispanic          0.207      0.211     0.006     0.001     -0.001   
Race:Other            -0.238      0.055     0.017       0.0     -0.011   
Educ:College           0.608     1.015*     0.001     0.003      0.007   
Educ:Postgrad          0.237      0.282     0.019     0.006     -0.001   
Educ:Some college      0.338      0.393     0.011     0.006      0.001   
Age:25-34               0.48      0.541     0.007     0.003      0.001   
Age:35-49           2.463***   2.639***    0.023*   0.019**    0.026**   
Age:50-64            2.58***   2.827***     0.013  0.023***   0.034***   
Age:65+             3.761***   4.125***  0.043***    0.02**   0.047***   
woman                 -0.123      0.02



=== Exposure rate, DESKTOP-ONLY (n=690) ===
                          Ad    Cookies Canvas FP    Session     Keylog  \
Intercept           4.198***    5.52***   0.06***   0.029***    0.029**   
Race:Asian         -1.588***  -1.636***  -0.025**  -0.018***  -0.033***   
Race:Black             0.806      1.095     0.008      0.012      0.017   
Race:Hispanic          0.317      0.325      0.01      0.003      0.008   
Race:Other             0.118      0.763     0.031      0.003     -0.009   
Educ:College           0.645        0.6    -0.004      0.008      0.001   
Educ:Postgrad          0.593      0.745     0.013       0.01      -0.01   
Educ:Some college      0.346      0.117      0.01      0.003     -0.002   
Age:25-34             -0.274     -0.574    -0.003     -0.006     -0.003   
Age:35-49           2.316***    2.315**     0.025      0.011   0.045***   
Age:50-64           2.291***   2.452***     0.008      0.009   0.045***   
Age:65+             3.681***   3.736***    0.032*     

In [6]:
print('=== Cumulative exposure, FULL ===')
print(coef_table(data_full, 'cum'))
print()
print('=== Cumulative exposure, DESKTOP-ONLY ===')
print(coef_table(data_desk, 'cum'))

=== Cumulative exposure, FULL ===


                             Ad       Cookies   Canvas FP     Session  \
Intercept          11049.841***  12611.195***     69.762*     74.04**   
Race:Asian             1429.476      3843.512      19.428   -58.426**   
Race:Black            -1887.884     -2799.152     -39.232      -3.453   
Race:Hispanic         -4095.816     -4214.525     -24.542     -19.925   
Race:Other            -1404.371      -308.258     191.648     -33.074   
Educ:College       12537.594***  15975.719***      86.442    87.468**   
Educ:Postgrad          8905.68*     8656.062*    160.182*     68.009*   
Educ:Some college       916.391      2706.163       32.11       4.558   
Age:25-34              2187.539      2442.784      36.959       0.798   
Age:35-49           9924.211***   9819.582***    84.402**      37.427   
Age:50-64          18623.657***  21818.922***  152.784***    75.598**   
Age:65+            30952.042***  35200.026***  358.778***  136.045***   
woman                 -3587.992     -3106.886    92

                             Ad       Cookies   Canvas FP      Session  \
Intercept          15913.442***  17544.921***       81.31    122.945**   
Race:Asian            -3493.335     -1056.827     -18.608  -101.603***   
Race:Black             2997.607      3239.251      -7.943        1.126   
Race:Hispanic         -5475.833     -4678.699        13.3      -40.571   
Race:Other             4688.554      7856.082    399.628*      -31.249   
Educ:College           6648.682      9961.921      14.397       75.401   
Educ:Postgrad          1278.738      -224.798      83.886       41.662   
Educ:Some college     -4076.652     -2142.557      -6.685      -33.093   
Age:25-34              5265.078      6056.265      91.705        11.76   
Age:35-49          22284.361***  23726.978***  244.902***       77.665   
Age:50-64           29501.68***  35253.434***  268.239***     105.513*   
Age:65+            36061.823***  41155.397***  423.317***   137.064***   
woman                  -811.961       